# APPRAISE · Rank two published nanobodies
Choose **Runtime → Run all** to remeasure ten published structures on CPU. The example compares Nb6B9 and Nb80 against activated β₂AR. It saves a ranked table and heatmap and normally places Nb6B9 first.

For fresh predictions on a free Tesla T4, choose AlphaFold2-multimer. Both engines completed the 554-residue guided case. ESMFold2-Fast remains experimental because the six-nanobody ranking missed the qualification gate. Estimated GPU charge is $0 when Google assigns a free runtime. Google bills optional paid Colab compute. [ESMFold2 cost estimates and billing](https://github.com/xz-ding/APPRAISE/blob/main/docs/nanobody-runner.md#estimated-esmfold2-usage-costs) · [Published experiment](https://pmc.ncbi.nlm.nih.gov/articles/PMC11184338/#fig3) · [Original structures](https://data.caltech.edu/records/kxjgj-tfk18)

In [ ]:
#@title Choose the example and checkpoint location
workflow = "Saved structures on CPU" #@param ["Saved structures on CPU", "Published six-nanobody measurements", "AlphaFold2-multimer (recommended GPU)", "ESMFold2-Fast (experimental)"]
panel_size = "Two nanobodies" #@param ["Two nanobodies", "Six nanobodies"]
save_to_drive = False #@param {type:"boolean"}
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

routes = {
    "Saved structures on CPU": ("b2ar-saved", "saved"),
    "Published six-nanobody measurements": ("b2ar-reference", "reference"),
    "AlphaFold2-multimer (recommended GPU)": ("b2ar-two", "colabfold"),
    "ESMFold2-Fast (experimental)": ("b2ar-two", "esmfold2-fast"),
}
example, engine = routes[workflow]
fresh = engine in {"esmfold2-fast", "colabfold"}
if fresh and panel_size == "Six nanobodies":
    example = "b2ar-six"
jobs = 36 if example in {"b2ar-reference", "b2ar-six"} else 2
print(f"{workflow} · {jobs} ordered inputs · {jobs * 5} structures · {jobs * 10} candidate measurements")
if fresh:
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Select a free GPU under Runtime → Change runtime type, or choose Saved structures on CPU.")
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.free,memory.total,compute_cap,driver_version",
                    "--format=csv"], check=True)
if save_to_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/APPRAISE/b2ar")
else:
    project = Path("/content/appraise-local/b2ar")
project.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=project, delete=False) as probe:
    probe.write(b"APPRAISE checkpoint")
    probe_path = Path(probe.name)
try:
    if probe_path.read_bytes() != b"APPRAISE checkpoint":
        raise OSError(f"Checkpoint read-back failed in {project}")
finally:
    probe_path.unlink(missing_ok=True)
run_dir = project / f"{example}-{engine}"
print(f"Results and checkpoints: {run_dir}")
if not save_to_drive:
    print("Local files last for this Colab session. Download the results before disconnecting. For runs that span sessions, use Google Drive.")

In [ ]:
#@title Install APPRAISE and the selected engine
release = "1.3.2"
env_dir = Path("/content") / f"appraise-pypi-venv-{engine}-{release}-py{sys.version_info.major}{sys.version_info.minor}"
python = env_dir / "bin/python"
setup_dir = project / "setup" / f"{engine}-{release}"
setup_dir.mkdir(parents=True, exist_ok=True)
log_path = setup_dir / "install.log"
install_start = time.monotonic()

def checked(args):
    with log_path.open("a", encoding="utf-8") as log:
        status = subprocess.run([str(x) for x in args], stdout=log, stderr=subprocess.STDOUT)
    if status.returncode:
        print(log_path.read_text(errors="replace")[-3500:])
        raise RuntimeError(f"Setup exited with code {status.returncode}. Full log: {log_path}")

if not python.exists():
    import shutil
    if shutil.which("uv"):
        checked(["uv", "venv", "--seed", "--system-site-packages", "--python", sys.executable, env_dir])
    else:
        checked([sys.executable, "-m", "venv", "--system-site-packages", env_dir])
req = "appraise" if engine == "reference" else "appraise[structures]"
checked([python, "-m", "pip", "install", "--quiet", "--index-url", "https://pypi.org/simple", f"{req}=={release}"])
if fresh:
    from importlib.metadata import version
    # Keep the GPU-enabled JAX and jaxlib supplied by this Colab runtime.
    constraints = env_dir / "colab-jax.txt"
    constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("jax", "jaxlib")))
    install = [python, "-m", "pip", "install", "--quiet", "-c", constraints]
    if engine == "colabfold":
        checked(install + ["colabfold[alphafold-minus-jax,openmm] @ git+https://github.com/sokrypton/ColabFold.git@84c27d9cc500489fd9b97545d2325b9d00f251d5"])
        checked([python, "-c", "from colabfold.batch import _tool_path; import shutil; names = ['hhsearch','kalign']; assert all(shutil.which(_tool_path(n)) for n in names), 'Install the ColabFold template binaries'; import openmm, pdbfixer"])
    else:
        if sys.version_info < (3, 12):
            raise RuntimeError("The pinned ESMFold2 preview requires a Colab runtime with Python 3.12 or newer.")
        checked(install + ["alphafold3-colabfold==3.1.20", "dm-haiku==0.0.17", "rdkit==2025.9.4",
                           "tokamax==0.0.11", "huggingface_hub", "zstandard"])
    checked([python, "-c", "import jax; print(jax.devices()); assert jax.default_backend() == 'gpu', 'JAX cannot see the GPU'"])

probe = "import json,sys; from importlib import metadata; from appraise.runner import RunConfig; d=metadata.distribution('appraise'); print(json.dumps({'version':d.version,'source':'PyPI' if d.read_text('direct_url.json') is None else 'direct URL','python':sys.version.split()[0]}))"
installed = json.loads(subprocess.check_output([python, "-c", probe], text=True))
if installed["source"] != "PyPI" or installed["version"] != release:
    raise RuntimeError("The installed APPRAISE distribution does not match this notebook. Rerun setup in a fresh session.")
installed["setup_seconds"] = round(time.monotonic() - install_start, 2)
installed["workflow"] = workflow
(setup_dir / "installation.json").write_text(json.dumps(installed, indent=2))
print(f"APPRAISE {installed['version']} · {installed['source']} · setup {installed['setup_seconds']} s")

In [ ]:
#@title Run or resume the competition
command = [str(python), "-m", "appraise"]
if (run_dir / "run_manifest.json").is_file():
    command += ["resume", "--run-dir", str(run_dir)]
elif fresh:
    command += ["run", "--example", example, "--engine", engine, "--output", str(run_dir)]
else:
    command += ["score", "--example", example, "--output", str(run_dir)]

import tempfile
with tempfile.TemporaryFile(mode="w+t", encoding="utf-8") as response:
    with subprocess.Popen(command, stdout=response, stderr=subprocess.PIPE, text=True) as process:
        for line in process.stderr:
            print(line, end="", flush=True)
    response.seek(0)
    try:
        result = json.load(response)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"APPRAISE exited with code {process.returncode} before writing a readable result. Check {run_dir / 'run.log'}.") from exc
print(f"{result['status']} · {result.get('completed_predictions', 0)} structures · {result.get('completed_measurements', 0)} measurements")
if result["status"] != "complete":
    raise RuntimeError(f"{result.get('stage')}: {result.get('error')} {result.get('next_action')} Files: {run_dir}")

In [ ]:
#@title View and download results
import pandas as pd
import ipywidgets as widgets
from IPython.display import Image, display
from google.colab import files
ranking = pd.read_csv(result["artifacts"]["ranking.csv"]["path"])
display(ranking[["rank", "peptide_name", "win_fraction", "Delta_B"]].rename(columns={
    "peptide_name": "Nanobody", "win_fraction": "Tournament win fraction", "Delta_B": "Mean ΔB"}))
display(Image(filename=result["artifacts"]["heatmap.png"]["path"]))
if "comparison" in result:
    comparison = result["comparison"]
    print(f"{comparison['evidence']}. Spearman ρ = {comparison['spearman_rho']}; ranking criterion met = {comparison['meets_ranking_criterion']}.")
def download_button(label, filename):
    button = widgets.Button(description=label)
    button.on_click(lambda _: files.download(str(run_dir / filename)))
    return button
display(widgets.HBox([download_button(label, name) for label, name in (
    ("Ranking CSV", "ranking.csv"), ("Heatmap PNG", "heatmap.png"), ("Measurements", "measurements.csv"))]))
display(widgets.HBox([download_button(label, name) for label, name in (
    ("Run log", "run.log"), ("Manifest", "run_manifest.json"), ("Result JSON", "result.json"))]))
print(run_dir)

Rank 1 has the strongest relative binding propensity in this panel. Equal ranks indicate a tie. Blue marks positive ΔB, red marks negative ΔB, and white marks values near zero. In the two-candidate example, gray diagonal cells mark unscheduled self-comparisons. The two directions together give ten independent structures per matchup. APPRAISE scores express relative competition within the supplied panel.

To resume a Drive run after a disconnect, select the same workflow and panel, enable `save_to_drive`, and run all cells. The six-candidate comparison requires 36 ordered inputs, including self-pairs, with five structures per input. [Advanced manual notebook](https://github.com/xz-ding/APPRAISE/blob/main/Colab_APPRAISE.ipynb)